# BLB Hebrew Pronunciation Audio Downloader + Analyser + Splitter

**Cell 1** — Config  
**Cell 2** — Setup (folders, cache)  
**Cell 3** — Helpers  
**Cell 4** — Download loop (resume-safe)  
**Cell 5** — Silence analysis → CSV  
**Cell 6** — Split / reformat audio with ffmpeg  

Re-running any cell is safe — already-done work is skipped.
# Load and Define

In [32]:
NO_ENTRIES = [] # entries that have "no entry" "not exists" instead of pronouncation

## Cell 1 — Config

In [2]:
import os
START      = 1        # first Strong's number
END        = 100      # last Strong's number (inclusive)

MUSIC_DIR   = "music_g"        # raw downloaded audio
HTML_DIR    = "strongs_g"    # cached lexicon HTML pages
SPLIT_DIR   = "strongs_p_g"    # final split/reformatted audio

HASH_CACHE_FILE   = f"{MUSIC_DIR}{os.sep}blb_pronunc_gk_hashes.json"
SILENCE_CSV_FILE  = f"{MUSIC_DIR}{os.sep}blb_silence_analysis_gk.csv"

DELAY_OK    = 0.15#1.5   # seconds between successful requests
DELAY_RETRY = 30    # base backoff seconds on rate-limit / timeout
MAX_RETRIES = 5

# ffmpeg silence detection thresholds
SILENCE_DB       = -37    # dB threshold — adjust if too many/few gaps detected
SILENCE_MIN_DUR  = 0.350    # minimum gap duration in seconds to count as silence
SILENCE_SPIKES_MERGE_DURATION_MAX = 0.200 #max lenght of noise spikes that occur between two silences and will be treated as silence instead
SILENCE_TRAILING_SILENCE_MAX_DURATION_TO_BE_MERGED_WITH_PREVIOUS_CHUNK = 0.4 #max length of trailing silence to be merged with previous chhung (speech)
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ 2 excepts on 200. now what if.
SILENCE_DB       = -50 

## Cell 2 — Setup

In [3]:
import os, json, time, re, shutil, subprocess
import urllib.request, urllib.error, urllib.parse
from pathlib import Path
import pandas as pd

for d in (MUSIC_DIR, HTML_DIR, SPLIT_DIR):
    Path(d).mkdir(parents=True, exist_ok=True)
    print(f"✅ Folder ready: {Path(d).resolve()}")

# ── hash cache ─────────────────────────────────────────────────────────────
cache_path = Path(HASH_CACHE_FILE)
if cache_path.exists():
    with open(cache_path, "r", encoding="utf-8") as f:
        hash_cache: dict = json.load(f)
    print(f"📦 Loaded hash cache ({len(hash_cache)} entries)")
else:
    hash_cache = {}
    print("📦 Fresh hash cache")

def save_cache():
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(hash_cache, f, indent=2)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

try:
    subprocess.run(["ffmpeg", "-version"], capture_output=True, check=True)
    print("✅ ffmpeg found")
except FileNotFoundError:
    print("❌ ffmpeg not found — install it before running cells 5 and 6")

✅ Folder ready: /home/u/g_bib_local/music_g
✅ Folder ready: /home/u/g_bib_local/strongs_g
✅ Folder ready: /home/u/g_bib_local/strongs_p_g
📦 Loaded hash cache (5624 entries)
✅ ffmpeg found


## Cell 3 — Helpers

In [4]:
# ── network ────────────────────────────────────────────────────────────────

def fetch_html(url: str, retries: int = MAX_RETRIES) -> str | None:
    for attempt in range(1, retries + 1):
        try:
            req = urllib.request.Request(url, headers=HEADERS)
            with urllib.request.urlopen(req, timeout=20) as resp:
                return resp.read().decode("utf-8", errors="replace")
        except urllib.error.HTTPError as e:
            if e.code in (429, 503):
                wait = DELAY_RETRY * attempt
                print(f"    ⏳ Rate-limited ({e.code}), waiting {wait}s (attempt {attempt}/{retries})...")
                time.sleep(wait)
            else:
                print(f"    ❌ HTTP {e.code} — {url}")
                return None
        except (urllib.error.URLError, TimeoutError, OSError) as e:
            wait = DELAY_RETRY * attempt
            print(f"    ⏳ Network error ({e}), waiting {wait}s (attempt {attempt}/{retries})...")
            time.sleep(wait)
    print(f"    ❌ Gave up after {retries} attempts: {url}")
    return None


def extract_pronunc_hash(html: str) -> str | None:
    m = re.search(r'data-pronunc="([^"]+)"', html)
    return m.group(1) if m else None


def audio_url(skin_hash: str) -> str:
    return (
        "https://www.blueletterbible.org/lang/lexicon/"
        f"lexPronouncePlayer.cfm?skin={skin_hash}"
    )

import requests
def download_audio(url: str, dest_dir: Path, strong_key: str,
                   retries: int = MAX_RETRIES) -> Path | None:
    session = requests.Session()
    session.headers.update(HEADERS)
    
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=30, stream=True)
            
            if resp.status_code in (429, 503):
                wait = DELAY_RETRY * attempt
                print(f"    ⏳ Rate-limited ({resp.status_code}), waiting {wait}s...")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            
            # get filename from Content-Disposition or fallback
            cd = resp.headers.get("Content-Disposition", "")
            fname_match = re.search(
                r'filename\*?=(?:UTF-8\'\'|"?)([^";\r\n]+)', cd, re.I
            )
            if fname_match:
                # server sends full server-side path like /mnt/nginx-server/.../h0001.mp3
                # take only the basename, just like curl -J does
                filename = Path(urllib.parse.unquote(fname_match.group(1).strip('"'))).name
            else:
                ct = resp.headers.get("Content-Type", "")
                ext = ".mp3" if "mpeg" in ct else ".ogg" if "ogg" in ct else ".audio"
                filename = f"{strong_key}{ext}"
            
            dest = dest_dir / filename
            with open(dest, "wb") as f:
                for chunk in resp.iter_content(chunk_size=8192):
                    f.write(chunk)
            return dest
            
        except requests.exceptions.HTTPError as e:
            print(f"    ❌ HTTP error: {e}")
            return None
        except requests.exceptions.RequestException as e:
            wait = DELAY_RETRY * attempt
            print(f"    ⏳ Request error ({e}), waiting {wait}s (attempt {attempt}/{retries})...")
            time.sleep(wait)
    
    print(f"    ❌ Gave up after {retries} attempts: {strong_key}")
    return None

# ── ffmpeg helpers ─────────────────────────────────────────────────────────
def detect_silences(audio_path: Path,
                    noise_db: int = SILENCE_DB,
                    min_dur: float = SILENCE_MIN_DUR,
                    total_dur: float = None,
                   spikes_merge_max: float = SILENCE_SPIKES_MERGE_DURATION_MAX) -> list[dict]:
    cmd = [
        "ffmpeg", "-i", str(audio_path),
        "-af", f"silencedetect=noise={noise_db}dB:d={min_dur}",
        "-f", "null", "-"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    output = result.stderr
    starts = re.findall(r"silence_start:\s*([\d.]+)", output)
    ends   = re.findall(r"silence_end:\s*([\d.]+)",   output)
    durs   = re.findall(r"silence_duration:\s*([\d.]+)", output)
    silences = [
        {"silence_start": float(s), "silence_end": float(e), "silence_dur": float(d)}
        for s, e, d in zip(starts, ends, durs)
    ]

    # merge gaps split by short noise spikes
    merged = []
    for s in silences:
        if merged and (s["silence_start"] - merged[-1]["silence_end"]) <= spikes_merge_max:
            merged[-1]["silence_end"] = s["silence_end"]
            merged[-1]["silence_dur"] = merged[-1]["silence_end"] - merged[-1]["silence_start"]
        else:
            merged.append(s.copy())

    # drop trailing silence at end of file
    if total_dur is not None:
        merged = [s for s in merged if s["silence_end"] < total_dur - SILENCE_TRAILING_SILENCE_MAX_DURATION_TO_BE_MERGED_WITH_PREVIOUS_CHUNK]

    return merged


def get_audio_duration(audio_path: Path) -> float | None:
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(audio_path)
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    try:
        return float(r.stdout.strip())
    except ValueError:
        return None
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ sceibing silences helper (List 2 Silences List)
def l2sl(l):
    if len(l) not in [1,3,5,7,9,11,13]:
        raise Exception(f"not supported length {len(l)}")
    answ=[]
    answ.append({'silence_start': 0.0, 'silence_end': -1, 'silence_dur': 0})
    answ.append({'silence_start': -1, 'silence_end': l[0], 'silence_dur': 0})
    answ.append({'silence_start': -1, 'silence_end': -1, 'silence_dur': 0})
    for i in range(1,len(l),2):
        answ.append({'silence_start': l[i], 'silence_end': -1, 'silence_dur': 0})
        answ.append({'silence_start': -1, 'silence_end': l[i+1], 'silence_dur': 0})
        answ.append({'silence_start': -1, 'silence_end': -1, 'silence_dur': 0})
    return answ


print("✅ Helpers ready")

✅ Helpers ready


# Do work()

## Cell 4 — Download loop (resume-safe)

Re-run freely — cached HTML and existing audio files are skipped.

In [ ]:
# will redownload audio files! (As there is no hash decoder that would allow to see which files are here..)
skipped_no_hash = []
skipped_dl_fail = []
downloaded      = []
already_had     = []

VERBOSE = False
html_root  = Path(HTML_DIR)
music_root = Path(MUSIC_DIR)
START=1
END=5624#2500
from tqdm.notebook import tqdm
for num in tqdm(range(START, END + 1)):
    key     = f"g{num}"
    html_fp = html_root / f"{key}.html"

    # ── STEP 1: get pronunc hash ───────────────────────────────────────────
    # Priority: in-memory cache → saved HTML on disk → live fetch
    if key in hash_cache:
        skin = hash_cache[key]
    elif html_fp.exists():
        with open(html_fp, "r", encoding="utf-8") as f:
            cached_html = f.read()
        skin = extract_pronunc_hash(cached_html)
        if(VERBOSE):
            print(f"[{num}/{END}] 📄 {key}: hash from saved HTML")
        hash_cache[key] = skin
        save_cache()
    else:
        lex_url = f"https://www.blueletterbible.org/lexicon/{key}/kjv/wlc/0-1/"
        if(VERBOSE):
            print(f"[{num}/{END}] 🌐 {key}: fetching {lex_url}")
        html = fetch_html(lex_url)
        if html is None:
            print(f"  ⚠️  Could not fetch page for {key}, skipping")
            skipped_dl_fail.append(key)
            continue
        # save HTML to disk
        with open(html_fp, "w", encoding="utf-8") as f:
            f.write(html)
        skin = extract_pronunc_hash(html)
        hash_cache[key] = skin
        save_cache()
        time.sleep(DELAY_OK)

    if not skin:
        print(f"[{num}/{END}] ⚪ {key}: no data-pronunc found — no audio")
        skipped_no_hash.append(key)
        continue

    # ── STEP 2: skip if any audio file for this key already exists ─────────
    existing = list(music_root.glob(f"{key}.*"))
    if existing:
        if(VERBOSE):
            print(f"[{num}/{END}] ✔️  {key}: already have {existing[0].name}")
        already_had.append(key)
        continue

    # ── STEP 3: download audio ─────────────────────────────────────────────
    aurl = audio_url(skin)
    if(VERBOSE):
        print(f"[{num}/{END}] 🎵 {key}: downloading...")
    dest = download_audio(aurl, music_root, key)
    if dest:
        if(VERBOSE):
            print(f"  ✅ Saved → {dest.name}")
        downloaded.append(str(dest))
    else:
        skipped_dl_fail.append(key)

    time.sleep(DELAY_OK)

print()
print("=" * 50)
print(f"✅ Newly downloaded : {len(downloaded)}")
print(f"✔️  Already had      : {len(already_had)}")
print(f"⚪ No audio         : {len(skipped_no_hash)}  {skipped_no_hash[:10]}")
print(f"❌ Failed           : {len(skipped_dl_fail)}  {skipped_dl_fail}")

  0%|          | 0/5624 [00:00<?, ?it/s]

## Cell 5 — Silence analysis → CSV

Runs `ffmpeg silencedetect` on every file in `music/`.  
**One row per silence gap** — so a file with 3 gaps produces 3 rows (plus metadata columns shared across all rows for that file).  
Re-running skips already-analysed keys.

### Inferred audio structure
```
Single-entry, one pronunciation variant:
  [H-number spoken]  GAP-0(long)  [pronunc]  (end)

Single-entry, two pronunciation variants:
  [H-number spoken]  GAP-0(long)  [pronunc A]  GAP-1(short)  [pronunc B]  (end)

Two lexicon entries:
  [H-number spoken]  GAP-0(long)  [pronunc 1A]  GAP-1(short?)  [pronunc 1B?]
  GAP-2(long)  ["second entry"]  GAP-3(long)  [pronunc 2A]  GAP-4(short?)  [pronunc 2B?]  (end)
```
Gap index (`silence_index`, 0-based) tells you position in the file.  
Longer gaps mark structural boundaries; shorter gaps separate pronunciation variants.

In [39]:
csv_path   = Path(SILENCE_CSV_FILE)
music_root = Path(MUSIC_DIR)
LIMIT_UPTO_ANALYZE_COUNT = -1#5500#1000
OFFSET_FROM_ANALYZE_COUNT = -1#499
PRINT_VERBOSE_LOG=False #True
PRINT_VERBOSE_VERBOSE_ALREADY_ANALYSED_LOG=False
#noise_db
        # "noise_db", SILENCE_DB),
        # "min_dur", SILENCE_MIN_DUR),
        # "spikes_merge_max", SILENCE_SPIKES_MERGE_DURATION_MAX),
SILENCE_EXCEPTIONS = {
}


# load existing CSV to skip already-analysed keys
if csv_path.exists():
    df_existing = pd.read_csv(csv_path)
    already_analysed = set(df_existing["key"].unique())
    print(f"📊 Loaded existing CSV ({len(df_existing)} rows, {len(already_analysed)} keys)")
else:
    df_existing = pd.DataFrame()
    already_analysed = set()
    print("📊 No existing CSV — starting fresh")

new_rows = []

# collect all audio files, sorted numerically
def sort_key(p):
    m = re.search(r"\d+", p.stem)
    return int(m.group()) if m else 0
    
audio_files = []
for ext in ("mp3", "ogg", "audio"):
    audio_files += sorted(music_root.glob(f"g*.{ext}"), key=sort_key)

from tqdm.notebook import tqdm
cnt = 0
for audio_fp in tqdm(audio_files):
    cnt=cnt+1
    if(LIMIT_UPTO_ANALYZE_COUNT>0):
        if(cnt>LIMIT_UPTO_ANALYZE_COUNT):
            break
    if(OFFSET_FROM_ANALYZE_COUNT>0):
        if(cnt<OFFSET_FROM_ANALYZE_COUNT):
            continue
    
    m = re.search(r"(g\d+)", audio_fp.stem, re.I)
    if not m:
        print(f"⚠️  Can't parse key from: {audio_fp.name} — skipping")
        continue
    key = m.group(1).lower()

    if key in already_analysed:
        if(PRINT_VERBOSE_VERBOSE_ALREADY_ANALYSED_LOG):
            print(f"✔️  {key}: already analysed")
        continue

    if(PRINT_VERBOSE_LOG):
        print(f"🔍 {key}: analysing {audio_fp.name}...", end=" ", flush=True)
        print(SILENCE_EXCEPTIONS.get(key, {}))
    total_dur = get_audio_duration(audio_fp)
    overrides = SILENCE_EXCEPTIONS.get(key, {})
    silences = detect_silences(
        audio_fp,
        total_dur=total_dur,
        noise_db=overrides.get("noise_db", SILENCE_DB),
        min_dur=overrides.get("min_dur", SILENCE_MIN_DUR),
        spikes_merge_max=overrides.get("spikes_merge_max", SILENCE_SPIKES_MERGE_DURATION_MAX),
    )
    if(len(silences)%3!=0 or not silences):
        if(PRINT_VERBOSE_LOG):
            print(f"RETRY!: {len(silences)} gap(s)  (total {total_dur:.2f}s)")
        if(PRINT_VERBOSE_LOG):
            print(f"🔍 {key}: analysing {audio_fp.name}...", end=" ", flush=True)
            print(SILENCE_EXCEPTIONS.get(key, {}))
        total_dur = get_audio_duration(audio_fp)
        overrides = SILENCE_EXCEPTIONS.get(key, {})
        silences = detect_silences(
            audio_fp,
            total_dur=total_dur,
            noise_db=overrides.get("noise_db", -30),
            min_dur=overrides.get("min_dur", 0.12),
            spikes_merge_max=overrides.get("spikes_merge_max", SILENCE_SPIKES_MERGE_DURATION_MAX),
        )
    if(PRINT_VERBOSE_LOG):
        print(f"{len(silences)} gap(s)  (total {total_dur:.2f}s)")

    base_row = {
        "key":           key,
        "filename":      audio_fp.name,
        "total_dur":     total_dur,
        "silence_count": len(silences),
    }

    if not silences:
        # one row with nulls for silence columns
        new_rows.append({**base_row,
                         "silence_index": None,
                         "silence_start": None,
                         "silence_end":   None,
                         "silence_dur":   None})
    else:
        for idx, s in enumerate(silences):
            new_rows.append({**base_row,
                             "silence_index": idx,
                             "silence_start": s["silence_start"],
                             "silence_end":   s["silence_end"],
                             "silence_dur":   s["silence_dur"]})

# save
if new_rows:
    df_new = pd.DataFrame(new_rows)
    df_all = pd.concat([df_existing, df_new], ignore_index=True)
    df_all.to_csv(csv_path, index=False)
    print(f"\n💾 Saved {len(df_all)} total rows → {csv_path}")
else:
    df_all = df_existing
    print("\nNo new files to analyse.")

# summary table
if not df_all.empty:
    summary = (
        df_all.groupby("key")["silence_count"]
        .first()
        .value_counts()
        .sort_index()
        .rename_axis("gap_count")
        .rename("files")
    )
    print("\nSilence-gap distribution:")
    print(summary.to_string())

📊 No existing CSV — starting fresh


  0%|          | 0/5522 [00:00<?, ?it/s]


💾 Saved 17953 total rows → music_g/blb_silence_analysis_gk.csv

Silence-gap distribution:
gap_count
2        1
3     5066
4       72
5        7
6      286
7       54
8        7
9       13
10      14
11       1
12       1


### 1 OVERRIDES 1 - 5s, 8s

In [34]:
import statistics

split_root = Path(SPLIT_DIR)
csv_path   = Path(SILENCE_CSV_FILE)
music_root = Path(MUSIC_DIR)
PRINT_VERBOSE_LOG=False #True
PRINT_VERBOSE_VERBOSE_ALREADY_ANALYSED_LOG=False

if csv_path.exists():
    df_existing = pd.read_csv(csv_path)
    already_analysed = set(df_existing["key"].unique())
    print(f"📊 Loaded existing CSV ({len(df_existing)} rows, {len(already_analysed)} keys)")
else:
    df_existing = pd.DataFrame()
    already_analysed = set()
    print("📊 No existing CSV — starting fresh")

new_rows = []

# collect all audio files, sorted numerically
def sort_key(p):
    m = re.search(r"\d+", p.stem)
    return int(m.group()) if m else 0
    
audio_files = []
for ext in ("mp3", "ogg", "audio"):
    audio_files += sorted(music_root.glob(f"g*.{ext}"), key=sort_key)
#simplier writing of the intervals - just provide time for last speech pair
#and before that begin, end of each of previous if such exist
def l2sl(l):
    if len(l) not in [1,3,5,7,9,11,13]:
        raise Exception(f"not supported length {len(l)}")
    answ=[]
    answ.append({'silence_start': 0.0, 'silence_end': -1, 'silence_dur': 0})
    answ.append({'silence_start': -1, 'silence_end': l[0], 'silence_dur': 0})
    answ.append({'silence_start': -1, 'silence_end': -1, 'silence_dur': 0})
    for i in range(1,len(l),2):
        answ.append({'silence_start': l[i], 'silence_end': -1, 'silence_dur': 0})
        answ.append({'silence_start': -1, 'silence_end': l[i+1], 'silence_dur': 0})
        answ.append({'silence_start': -1, 'silence_end': -1, 'silence_dur': 0})
    return answ

to_reanalyse = ['g2661', # 2
                'g0281', 'g0293', 'g0390', 'g1207', 'g1308', 'g2085', 'g4379', # 5
                'g1155', 'g1157', 'g1257', 'g1524', 'g1527', 'g1696', 'g1944' # 8
               ]
df_existing = df_existing[~df_existing['key'].isin(to_reanalyse)]

from tqdm.notebook import tqdm
for audio_fp in tqdm(audio_files):
    m = re.search(r"(g\d+)", audio_fp.stem, re.I)
    if not m:
        print(f"⚠️  Can't parse key from: {audio_fp.name} — skipping")
        continue
    key = m.group(1).lower()
    
    if key in NO_ENTRIES:
        if(PRINT_VERBOSE_LOG):
            print(f"⏩ {key} marked as no entry. skipping")
        continue  
           
    if key not in to_reanalyse:
            continue
    silences =[]
    if(key=="g2661"):
        silences=l2sl([
            3.535
        ])
    if(key=="g0281"):
        silences=l2sl([
            4.266
        ])
    if(key=="g0293"):
        silences=l2sl([
            4.251
        ])
    if(key=="g0390"):
        silences=l2sl([
            3.823
        ])
    if(key=="g1207"):
        silences=l2sl([
            3.896
        ])
    if(key=="g1308"):
        silences=l2sl([
            4.619
        ])
    if(key=="g2085"):
        silences=l2sl([
            4.679
        ])
    if(key=="g4379"):
        silences=l2sl([
            3.958
        ])
    # 8-ish
    if(key=="g1155"):
        silences=l2sl([
            4.577, 7.287, 11.710
        ])
    if(key=="g1157"):
        silences=l2sl([
            4.798, 7.869, 11.924
        ])
    if(key=="g1257"):
        silences=l2sl([
            4.347, 7.041, 11.263
        ])
    if(key=="g1524"):
        silences=l2sl([
            4.686, 6.917, 10.881
        ])
    if(key=="g1527"):
        silences=l2sl([
            4.724, 8.560, 11.133
        ])
    if(key=="g1696"):
        silences=l2sl([
            4.926, 7.171, 11.444
        ])
    if(key=="g1944"):
        silences=l2sl([
            3.890, 7.523, 12.362
        ])

    for i in range(len(silences)):
        silences[i]['silence_dur'] = round(silences[i]['silence_end']-silences[i]['silence_start'], 6)
    if Path(split_root / f"{key}.mp3").exists():
        Path(split_root / f"{key}.mp3").unlink()
    for g in range(2,8):
        if Path(split_root / f"{key}-{g}.mp3").exists():
            Path(split_root / f"{key}-{g}.mp3").unlink()  
    total_duration = get_audio_duration(audio_fp)
    #total_duration = df_existing[df_existing['key'] == key].iloc[0]['total_dur']
    df_existing = df_existing[~df_existing['key'].isin([key])]
    base_row = {
        "key":           key,
        "filename":      audio_fp.name,
        "total_dur":     total_duration,
        "silence_count": len(silences),
    }
    
    for idx, s in enumerate(silences):
        new_rows.append({**base_row,\
                         "silence_index": idx,
                         "silence_start": s["silence_start"],
                         "silence_end":   s["silence_end"],
                         "silence_dur":   s["silence_dur"]})

# save
if new_rows:
    df_new = pd.DataFrame(new_rows)
    df_all = pd.concat([df_existing, df_new], ignore_index=True)
    df_all.to_csv(csv_path, index=False)
    print(f"\n💾 Saved {len(df_all)} total rows → {csv_path}")
else:
    df_all = df_existing
    print("\nNo new files to analyse.")

# summary table
if not df_all.empty:
    summary = (
        df_all.groupby("key")["silence_count"]
        .first()
        .value_counts()
        .sort_index()
        .rename_axis("gap_count")
        .rename("files")
    )
    print("\nSilence-gap distribution:")
    print(summary.to_string())

📊 Loaded existing CSV (17953 rows, 5522 keys)


  0%|          | 0/5522 [00:00<?, ?it/s]


💾 Saved 17926 total rows → music_g/blb_silence_analysis_gk.csv

Silence-gap distribution:
gap_count
3     5074
4       72
6      293
7       54
9       13
10      14
11       1
12       1


### 2 OVERRIDES 2 -10s 11s

In [35]:
import statistics

split_root = Path(SPLIT_DIR)
csv_path   = Path(SILENCE_CSV_FILE)
music_root = Path(MUSIC_DIR)
PRINT_VERBOSE_LOG=False #True
PRINT_VERBOSE_VERBOSE_ALREADY_ANALYSED_LOG=False

if csv_path.exists():
    df_existing = pd.read_csv(csv_path)
    already_analysed = set(df_existing["key"].unique())
    print(f"📊 Loaded existing CSV ({len(df_existing)} rows, {len(already_analysed)} keys)")
else:
    df_existing = pd.DataFrame()
    already_analysed = set()
    print("📊 No existing CSV — starting fresh")

new_rows = []

# collect all audio files, sorted numerically
def sort_key(p):
    m = re.search(r"\d+", p.stem)
    return int(m.group()) if m else 0
    
audio_files = []
for ext in ("mp3", "ogg", "audio"):
    audio_files += sorted(music_root.glob(f"g*.{ext}"), key=sort_key)
#simplier writing of the intervals - just provide time for last speech pair
#and before that begin, end of each of previous if such exist
def l2sl(l):
    if len(l) not in [1,3,5,7,9,11,13]:
        raise Exception(f"not supported length {len(l)}")
    answ=[]
    answ.append({'silence_start': 0.0, 'silence_end': -1, 'silence_dur': 0})
    answ.append({'silence_start': -1, 'silence_end': l[0], 'silence_dur': 0})
    answ.append({'silence_start': -1, 'silence_end': -1, 'silence_dur': 0})
    for i in range(1,len(l),2):
        answ.append({'silence_start': l[i], 'silence_end': -1, 'silence_dur': 0})
        answ.append({'silence_start': -1, 'silence_end': l[i+1], 'silence_dur': 0})
        answ.append({'silence_start': -1, 'silence_end': -1, 'silence_dur': 0})
    return answ

to_reanalyse = ['g1888', 'g3569', 'g5023', 'g5025', 'g5026', 'g5104', 'g5120', 'g5124', # 10
                'g5125', 'g5126', 'g5127', 'g5128', 'g5129', 'g5130', # 10
                'g1440' # 11
               ]
df_existing = df_existing[~df_existing['key'].isin(to_reanalyse)]

from tqdm.notebook import tqdm
for audio_fp in tqdm(audio_files):
    m = re.search(r"(g\d+)", audio_fp.stem, re.I)
    if not m:
        print(f"⚠️  Can't parse key from: {audio_fp.name} — skipping")
        continue
    key = m.group(1).lower()
    
    if key in NO_ENTRIES:
        if(PRINT_VERBOSE_LOG):
            print(f"⏩ {key} marked as no entry. skipping")
        continue  
           
    if key not in to_reanalyse:
            continue
    # 10 first row
    if(key=="g1888"):
        silences=l2sl([
            4.160, 7.438, 10.946, 14.077, 17.207
        ])
    if(key=="g3569"):
        silences=l2sl([
            4.095, 6.524, 9.485
        ])
    if(key=="g5023"):
        silences=l2sl([
            4.170, 6.200, 9.382
        ])
    if(key=="g5025"):
        silences=l2sl([
            4.187, 6.850, 9.825
        ])
    if(key=="g5026"):
        silences=l2sl([
            4.545, 6.727, 9.648
        ])
    if(key=="g5104"):
        silences=l2sl([
            3.593, 5.496, 8.662
        ])
    if(key=="g5120"):
        silences=l2sl([
            3.521, 5.274, 8.357
        ])
    if(key=="g5124"):
        silences=l2sl([
            3.968, 5.918, 8.928
        ])
    # 10 2nd row
    if(key=="g5125"):
        silences=l2sl([
            4.094, 6.523, 9.454
        ])
    if(key=="g5126"):
        silences=l2sl([
            4.348, 6.317, 9.210
        ])
    if(key=="g5127"):
        silences=l2sl([
            4.608, 6.513, 9.475
        ])
    if(key=="g5128"):
        silences=l2sl([
            3.802, 6.284, 9.258
        ])
    if(key=="g5129"):
        silences=l2sl([
            4.147, 6.044, 8.998
        ])
    if(key=="g5130"):
        silences=l2sl([
            3.748, 5.655, 8.673
        ])
    #11
    if(key=="g1440"):
        silences=l2sl([
            4.323, 7.547, 11.823, 15.620, 20.684
        ])

    for i in range(len(silences)):
        silences[i]['silence_dur'] = round(silences[i]['silence_end']-silences[i]['silence_start'], 6)
    if Path(split_root / f"{key}.mp3").exists():
        Path(split_root / f"{key}.mp3").unlink()
    for g in range(2,8):
        if Path(split_root / f"{key}-{g}.mp3").exists():
            Path(split_root / f"{key}-{g}.mp3").unlink()  
    total_duration = get_audio_duration(audio_fp)
    #total_duration = df_existing[df_existing['key'] == key].iloc[0]['total_dur']
    df_existing = df_existing[~df_existing['key'].isin([key])]
    base_row = {
        "key":           key,
        "filename":      audio_fp.name,
        "total_dur":     total_duration,
        "silence_count": len(silences),
    }
    for idx, s in enumerate(silences):
        new_rows.append({**base_row,\
                         "silence_index": idx,
                         "silence_start": s["silence_start"],
                         "silence_end":   s["silence_end"],
                         "silence_dur":   s["silence_dur"]})

# save
if new_rows:
    df_new = pd.DataFrame(new_rows)
    df_all = pd.concat([df_existing, df_new], ignore_index=True)
    df_all.to_csv(csv_path, index=False)
    print(f"\n💾 Saved {len(df_all)} total rows → {csv_path}")
else:
    df_all = df_existing
    print("\nNo new files to analyse.")

# summary table
if not df_all.empty:
    summary = (
        df_all.groupby("key")["silence_count"]
        .first()
        .value_counts()
        .sort_index()
        .rename_axis("gap_count")
        .rename("files")
    )
    print("\nSilence-gap distribution:")
    print(summary.to_string())

📊 Loaded existing CSV (17926 rows, 5522 keys)


  0%|          | 0/5522 [00:00<?, ?it/s]


💾 Saved 17871 total rows → music_g/blb_silence_analysis_gk.csv

Silence-gap distribution:
gap_count
3     5074
4       72
6      306
7       54
9       15
12       1


## Ad Hoc selects

In [25]:
df_all.query(
    #"silence_count==10 & key != 'h0969' & key != 'h1523'"
    "silence_count==8"
)["key"].unique()

<StringArray>
[]
Length: 0, dtype: str

## Ad hoc try auto ffmpeg

In [36]:
nm = "g0090"
audio_fp=os.path.join(MUSIC_DIR, f"{nm}.mp3")
total_dur = get_audio_duration(audio_fp)
silences =detect_silences(
        audio_fp,
        total_dur=total_dur,
        noise_db=-37,
        min_dur=0.35,
        spikes_merge_max=0.2,
    )
print(nm)
#print(json.dumps(silences, indent=1
print(len(silences))

g0090
7


## Ad hoc code

In [21]:
dels = []
for i in Path(os.path.join(f"{MUSIC_DIR}", "bad")).glob("*.audio"):
    dels.append(i.name.split(".")[0])
df_all = df_all[~df_all['key'].isin(dels)]
df_all.to_csv(csv_path, index=False)

In [16]:
Path(os.path.join("MUSIC_DIR", "bad"))

PosixPath('MUSIC_DIR/bad')

In [26]:
nu=df_all.query(
    "silence_count==7"
)
len(nu)

406

In [27]:
nu[:10]

,key,filename,total_dur,silence_count,silence_index,silence_start,silence_end,silence_dur
274,g0090,g0090.mp3,13.00900,7,0.0,0.000000,1.027256,1.027256
275,g0090,g0090.mp3,13.00900,7,1.0,2.269320,2.770181,0.500862
276,g0090,g0090.mp3,13.00900,7,2.0,3.817800,4.817868,1.000068
277,g0090,g0090.mp3,13.00900,7,3.0,5.910068,6.360476,0.450408
278,g0090,g0090.mp3,13.00900,7,4.0,7.600930,8.036735,0.435805
279,g0090,g0090.mp3,13.00900,7,5.0,9.125079,10.125057,0.999977
280,g0090,g0090.mp3,13.00900,7,6.0,11.039274,12.039274,1.000000
491,g0157,g0157.mp3,13.40075,7,0.0,0.000000,1.025057,1.025057
492,g0157,g0157.mp3,13.40075,7,1.0,3.412245,4.412245,1.000000
493,g0157,g0157.mp3,13.40075,7,2.0,5.118912,6.118934,1.000023


In [68]:
df = pd.read_csv("blb_silence_analysis.csv")
df = df[df["key"] != "h0067"]
df.to_csv("blb_silence_analysis.csv", index=False)
df = pd.read_csv("blb_silence_analysis.csv")

In [57]:
df_all = pd.read_csv("blb_silence_analysis.csv")

for n_gaps in [4, 5]:
    keys = df_all[df_all["silence_count"] == n_gaps]["key"].unique()
    for key in keys:
        rows = df_all[df_all["key"] == key].dropna(subset=["silence_start"])
        print(f"\n{'~'*50}")
        print(f"{key} ({n_gaps} gaps), total_dur={rows.iloc[0]['total_dur']:.2f}s")
        for _, r in rows.iterrows():
            print(f"  gap {int(r['silence_index'])}: {r['silence_start']:.2f}→{r['silence_end']:.2f}  (dur={r['silence_dur']:.2f}s)")

# and a couple of the 6-gap ones to confirm they are genuine 2-entry files
print("\n=== sample of 6-gap files ===")
keys_6 = df_all[df_all["silence_count"] == 6]["key"].unique()[:3]
for key in keys_6:
    rows = df_all[df_all["key"] == key].dropna(subset=["silence_start"])
    print(f"\n{key} (6 gaps), total_dur={rows.iloc[0]['total_dur']:.2f}s")
    for _, r in rows.iterrows():
        print(f"  gap {int(r['silence_index'])}: {r['silence_start']:.2f}→{r['silence_end']:.2f}  (dur={r['silence_dur']:.2f}s)")


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
h0067 (5 gaps), total_dur=10.76s
  gap 0: 0.00→1.03  (dur=1.03s)
  gap 1: 3.50→4.54  (dur=1.04s)
  gap 2: 5.32→5.67  (dur=0.35s)
  gap 3: 7.13→8.13  (dur=1.00s)
  gap 4: 8.92→9.33  (dur=0.41s)

=== sample of 6-gap files ===

h0074 (6 gaps), total_dur=13.53s
  gap 0: 0.00→1.05  (dur=1.05s)
  gap 1: 3.15→4.28  (dur=1.14s)
  gap 2: 5.17→6.21  (dur=1.04s)
  gap 3: 7.27→7.86  (dur=0.59s)
  gap 4: 9.83→10.36  (dur=0.53s)
  gap 5: 11.39→12.46  (dur=1.07s)

h0084 (6 gaps), total_dur=12.38s
  gap 0: 0.00→1.03  (dur=1.03s)
  gap 1: 3.06→4.14  (dur=1.07s)
  gap 2: 4.83→5.88  (dur=1.05s)
  gap 3: 6.51→7.22  (dur=0.71s)
  gap 4: 9.19→9.66  (dur=0.47s)
  gap 5: 10.54→11.57  (dur=1.03s)

h0130 (6 gaps), total_dur=12.02s
  gap 0: 0.00→1.04  (dur=1.04s)
  gap 1: 2.60→3.62  (dur=1.02s)
  gap 2: 4.36→5.39  (dur=1.02s)
  gap 3: 6.18→6.92  (dur=0.74s)
  gap 4: 8.89→9.44  (dur=0.54s)
  gap 5: 10.18→11.20  (dur=1.02s)


In [34]:
def detect_silences_raw(audio_path, noise_db=SILENCE_DB, min_dur=SILENCE_MIN_DUR):
    cmd = [
        "ffmpeg", "-i", str(audio_path),
        "-af", f"silencedetect=noise={noise_db}dB:d={min_dur}",
        "-f", "null", "-"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stderr)  # print everything so we see raw output

detect_silences_raw(Path("music") / "h0188.mp3")

ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 14.3.0 (conda-forge gcc 14.3.0-18)
  configuration: --prefix=/home/u/miniforge3/envs/py --cc=/home/conda/feedstock_root/build_artifacts/ffmpeg_1773007661644/_build_env/bin/x86_64-conda-linux-gnu-cc --cxx=/home/conda/feedstock_root/build_artifacts/ffmpeg_1773007661644/_build_env/bin/x86_64-conda-linux-gnu-c++ --nm=x86_64-conda-linux-gnu-nm --ar=x86_64-conda-linux-gnu-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-alsa --enable-libpulse --enable-libvpl --enable-vaapi --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib --enable-libv

In [59]:
df = pd.read_csv("blb_silence_analysis.csv")
df = df[df["key"] != "h0067"]
df.to_csv("blb_silence_analysis.csv", index=False)

In [30]:
test = Path("music") / "h0188.mp3"
print("total dur:", get_audio_duration(test))
print("with current settings:", detect_silences(test))
print("more aggressive -30dB:", detect_silences(test, noise_db=-30))
print("very aggressive -20dB:", detect_silences(test, noise_db=-20))

total dur: 6.47825
with current settings: [{'silence_start': 0.0, 'silence_end': 1.032698, 'silence_dur': 1.032698}, {'silence_start': 3.364603, 'silence_end': 5.929705, 'silence_dur': 2.5651020000000004}]
more aggressive -30dB: [{'silence_start': 0.0, 'silence_end': 1.03424, 'silence_dur': 1.03424}, {'silence_start': 3.342018, 'silence_end': 5.931361, 'silence_dur': 2.589343}]
very aggressive -20dB: [{'silence_start': 0.0, 'silence_end': 1.045646, 'silence_dur': 1.045646}, {'silence_start': 3.258186, 'silence_end': 5.93381, 'silence_dur': 2.6756240000000004}]


In [31]:
# extract just that suspected-speech segment to listen to it
ffmpeg_extract(Path("music") / "h0188.mp3", 4.401, 4.874, Path("music") / "h0188_suspect.mp3")

## Cell 6 — Split & reformat with ffmpeg

### Segment assignment logic

```
speech_seg[0]           → ALWAYS DROP  (speaker says the Strong's number)
speech_seg[1]           → entry-1, pronunciation A          → h<N>.mp3
speech_seg[2] (if any)  → depends on gap before it:
    SHORT gap            → entry-1, pronunciation B (concat with A) → h<N>.mp3
    LONG gap             → entry-2 preamble ("second entry") → DROP
speech_seg[3] (if any)  → entry-2, pronunciation A           → h<N>-2.mp3
...and so on.
```

**Short vs long** gap heuristic: threshold = 70% of median gap duration per file.  
Gaps >= threshold are "long" (structural boundary); below are "short" (variant separator).

Output in `strongs_p/`:
- `h1.mp3`   — entry 1 (all pronunciation variants concatenated if >1)
- `h1-2.mp3` — entry 2 (if a second lexicon entry exists)

Re-running skips already-split keys.

In [40]:
import statistics

split_root = Path(SPLIT_DIR)
music_root = Path(MUSIC_DIR)
csv_path   = Path(SILENCE_CSV_FILE)

if not csv_path.exists():
    raise RuntimeError("Run Cell 5 first to generate the silence CSV.")

df_all = pd.read_csv(csv_path)


# ── ffmpeg extract/concat helpers ──────────────────────────────────────────

def ffmpeg_extract(src: Path, start: float, end: float, dest: Path):
    """Cut [start, end) seconds from src → dest (mp3)."""
    cmd = [
        "ffmpeg", "-y",
        "-i", str(src),
        "-ss", f"{start:.6f}",
        "-t",  f"{end - start:.6f}",
        "-c:a", "libmp3lame", "-q:a", "2",
        str(dest)
    ]
    subprocess.run(cmd, capture_output=True, check=True)


def ffmpeg_concat(parts: list[Path], dest: Path):
    """Concatenate audio segments into dest."""
    if len(parts) == 1:
        shutil.copy(parts[0], dest)
        return
    list_file = dest.parent / f"_cl_{dest.stem}.txt"
    with open(list_file, "w") as f:
        for p in parts:
            f.write(f"file '{p.resolve()}'\n")
    cmd = [
        "ffmpeg", "-y",
        "-f", "concat", "-safe", "0",
        "-i", str(list_file),
        "-c:a", "libmp3lame", "-q:a", "2",
        str(dest)
    ]
    subprocess.run(cmd, capture_output=True, check=True)
    list_file.unlink(missing_ok=True)


# ── per-key splitting ──────────────────────────────────────────────────────
from tqdm.notebook import tqdm
def split_key(key: str, df: pd.DataFrame) -> list[Path]:
    rows = df[df["key"] == key].sort_values("silence_index")
    if rows.empty:
        return []

    src_name = rows.iloc[0]["filename"]
    src      = music_root / src_name
    if not src.exists():
        print(f"  ⚠️  Source not found: {src}")
        return []

    total_dur  = float(rows.iloc[0]["total_dur"])
    n_silences = int(rows.iloc[0]["silence_count"])

    if n_silences == 0:
        out = split_root / f"{key}.mp3"
        shutil.copy(src, out)
        return [out]

    sil_rows = rows.dropna(subset=["silence_start"]).sort_values("silence_index")
    silences = list(zip(
        sil_rows["silence_start"].astype(float),
        sil_rows["silence_end"].astype(float),
        sil_rows["silence_dur"].astype(float),
    ))

    # number of entries = gap_count / 3  (always divisible by 3)
    n_entries = n_silences // 3

    # for entry N (0-based):
    #   content starts at: end of gap (N*3 + 1)
    #   content ends at:   start of gap (N*3 + 2)  ... which gives [pronunc → gap → repeat]
    #   gap (N*3 + 2) end leads into next entry's announcement (or end of file)
    # but we want to INCLUDE gap-2 (the silence between pronunc and repeat) in the output
    # so: start = end of gap(N*3+1), end = start of gap(N*3+3) or total_dur
    if n_entries == 7:
        saved = []
        content_start = silences[1][1]
        content_end   = silences[3][0]
        out_name = f"{key}.mp3"
        out_path = split_root / out_name
        ffmpeg_extract(src, content_start, content_end, out_path)
        saved.append(out_path)

        out_name = f"{key}-{1 + 1}.mp3"
        out_path = split_root / out_name
        content_start = silences[5][1]
        content_end   = total_dur
        ffmpeg_extract(src, content_start, content_end, out_path)
        saved.append(out_path)

        return saved

    saved = []
    for n in tqdm(range(n_entries), leave=False):
        i_start = n * 3 + 1
        if n_silences==4:
            i_start = 2
        i_end   = n * 3 + 3
        content_start = silences[i_start][1]                          # end of gap
        content_end   = silences[i_end][0] if i_end < len(silences) else total_dur  # start of gap or EOF
        if n_silences ==4:
            content_end = total_dur
            
        out_name = f"{key}.mp3" if n == 0 else f"{key}-{n + 1}.mp3"
        out_path = split_root / out_name
        ffmpeg_extract(src, content_start, content_end, out_path)
        saved.append(out_path)
    return saved


# ── run over all valid analysed keys ─────────────────────────────────────────────

valid_keys = (
    df_all.groupby("key")["silence_count"]
    .first()
    .pipe(lambda s: s[((s % 3 == 0) & (s != 0)) | (s==4) | (s==7)])
    .index
)

keys_in_csv = sorted(
    valid_keys,
    key=lambda k: int(re.search(r"\d+", k).group())
)
from tqdm.notebook import tqdm
for key in tqdm(keys_in_csv):
    if (split_root / f"{key}.mp3").exists():
        if(PRINT_VERBOSE_VERBOSE_ALREADY_ANALYSED_LOG):
            print(f"✔️  {key}: already split")
        continue

    try:
        saved = split_key(key, df_all)
        if saved:
            if(PRINT_VERBOSE_VERBOSE_ALREADY_ANALYSED_LOG):
                print(f"✅ {key}: → {', '.join(p.name for p in saved)}")
        else:
            print(f"⚠️  {key}: nothing produced")
    except subprocess.CalledProcessError as e:
        stderr_tail = e.stderr[-300:] if e.stderr else ""
        print(f"❌ {key}: ffmpeg error — {stderr_tail}")
    except Exception as e:
        print(f"❌ {key}: {e}")
l=[]
for f in Path(SPLIT_DIR).glob("*.mp3"):
    l.append(f.name)
sdf = pd.DataFrame(l, columns=["filename"])
sdf.to_csv("ci/mp3list.csv", index=False)
sdf.to_csv("../bib_local/ci/mp3list_g.csv", index=False)
print("\n🏁 Done.")

  0%|          | 0/5522 [00:00<?, ?it/s]


🏁 Done.


In [ ]:
keys = ["h0074", "h0084" ]
for key in keys:
    print("~"*50)
    print(key)
    rows = df_all[df_all["key"] == key].sort_values("silence_index")
    sil_rows = rows.dropna(subset=["silence_start"]).sort_values("silence_index")
    silences = list(zip(
        sil_rows["silence_start"].astype(float),
        sil_rows["silence_end"].astype(float),
        sil_rows["silence_dur"].astype(float),
    ))
    total_dur = float(rows.iloc[0]["total_dur"])
    n_silences = int(rows.iloc[0]["silence_count"])
    n_entries = n_silences // 3
    
    print(f"n_silences={n_silences}, n_entries={n_entries}, total_dur={total_dur}")
    for i, (s, e, d) in enumerate(silences):
        print(f"  gap {i}: {s:.2f}→{e:.2f} (dur={d:.2f})")
    print()
    for n in range(n_entries):
        i_start = n * 3 + 1
        i_end   = n * 3 + 3
        cs = silences[i_start][1]
        ce = silences[i_end][0] if i_end < len(silences) else total_dur
        print(f"  entry {n}: start=silences[{i_start}].end={cs:.2f}  end=silences[{i_end}].start={ce:.2f} (idx exists: {i_end < len(silences)})")